In [33]:
import pandas as pd
import numpy as np
import os, pdb, sys, pickle, openai
api_key = 'your api key'

tr = pd.read_csv('/Users/christophershultz/Desktop/LLM_RISK_INSURANCE/reference_data/peril.training.csv')
te = pd.read_csv('/Users/christophershultz/Desktop/LLM_RISK_INSURANCE/reference_data/peril.validation.csv')
tr.head()

,Vandalism,Fire,Lightning,Wind,Hail,Vehicle,WaterNW,WaterW,Misc,Loss,Description
0,0,0,1,0,0,0,0,0,0,6838.87,lightning damage ...
1,0,0,1,0,0,0,0,0,0,2085.00,lightning damage at Comm. Center ...
2,0,0,1,0,0,0,0,0,0,11335.00,lightning damage at water tower ...
3,0,0,1,0,0,0,0,0,0,1480.00,lightning damge to radio tower ...
4,1,0,0,0,0,0,0,0,0,600.00,vandalism damage at recycle center ...


In [34]:
with open('dct2cat.pkl', 'rb') as f: 
    dct2cat = pickle.load(f)

with open('dct3cat.pkl', 'rb') as f: 
    dct3cat = pickle.load(f)

with open('dct5cat.pkl', 'rb') as f: 
    dct5cat = pickle.load(f)

In [35]:
tr['label_2cat'] = tr['Description'].map(dct2cat)
te['label_2cat'] = te['Description'].map(dct2cat)
tr['label_3cat'] = tr['Description'].map(dct3cat)
te['label_3cat'] = te['Description'].map(dct3cat)
tr['label_5cat'] = tr['Description'].map(dct5cat)
te['label_5cat'] = te['Description'].map(dct5cat)

for col in ['label_2cat', 'label_3cat', 'label_5cat']:
    tr[col] = [i.lower().replace("'", "") for i in tr[col]]
    te[col] = [i.lower().replace("'", "") for i in te[col]]


In [36]:
# Clean 2cat
tr['label_2cat'] = [i if i in ['low', 'high'] else '' for i in tr['label_2cat']]
te['label_2cat'] = [i if i in ['low', 'high'] else '' for i in te['label_2cat']]

# Clean 3cat
tr['label_3cat'] = [i if i in ['low', 'med', 'high'] else '' for i in tr['label_3cat']]
te['label_3cat'] = [i if i in ['low', 'med', 'high'] else '' for i in te['label_3cat']]

# Clean 5cat
tr['label_5cat'] = [i if i in ['1', '2', '3', '4', '5'] else '' for i in tr['label_5cat']]
te['label_5cat'] = [i if i in ['1', '2', '3', '4', '5'] else '' for i in te['label_5cat']]


In [37]:
tr.to_csv('peril_train_with_labels.csv', index=False)
te.to_csv('peril_test_with_labels.csv', index=False)

# Examine Averages

In [45]:
sub2 = tr[['label_2cat', 'Loss']].groupby(['label_2cat']).count().reset_index()
sub2 = sub2[sub2['label_2cat'] != ''].reset_index(drop=True)
sub2.columns = ['label_2cat', 'count']

sub2m = tr[['label_2cat', 'Loss']].groupby(['label_2cat']).mean().reset_index()
sub2 = pd.merge(sub2, sub2m, on='label_2cat', how='left')
sub2

,label_2cat,count,Loss
0,high,2085,38846.043165
1,low,2892,5271.794647


In [46]:
sub3 = tr[['label_3cat', 'Loss']].groupby(['label_3cat']).count().reset_index()
sub3 = sub3[sub3['label_3cat'] != ''].reset_index(drop=True)
sub3.columns = ['label_3cat', 'count']
sub3m = tr[['label_3cat', 'Loss']].groupby(['label_3cat']).mean().reset_index()
sub3 = pd.merge(sub3, sub3m, on='label_3cat', how='left')
sub3

,label_3cat,count,Loss
0,high,295,65801.234542
1,low,1051,2611.919134
2,med,3637,20383.396412


In [47]:
sub5 = tr[['label_5cat', 'Loss']].groupby(['label_5cat']).count().reset_index()
sub5 = sub5[sub5['label_5cat'] != ''].reset_index(drop=True)
sub5.columns = ['label_5cat', 'count']
sub5m = tr[['label_5cat', 'Loss']].groupby(['label_5cat']).mean().reset_index()
sub5 = pd.merge(sub5, sub5m, on='label_5cat', how='left')
sub5


,label_5cat,count,Loss
0,1,507,2218.732446
1,2,2654,6404.775170
2,3,1487,38755.755380
3,4,192,35496.178177
4,5,119,113383.592437
